Las librerías necesarias se instalan silenciosamente para mantener la claridad del documento.


In [1]:
def quiet_pip_install(package):
    import subprocess
    subprocess.run(["pip", "install"] + package.split(), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Instalar sin mostrar mensajes
quiet_pip_install("mapclassify")
quiet_pip_install("optbinning")
quiet_pip_install("mplleaflet")
quiet_pip_install("catboost")
quiet_pip_install("streamlit pyngrok joblib")
quiet_pip_install("pwlf")


In [2]:
#Importo librerias necesarias para el proceso de EDA (Exploratory Data Analysis)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
from matplotlib.colors import Normalize
import matplotlib.cm as cm
import seaborn as sns
import folium
import numpy as np
from datetime import datetime
import shutil
from google.colab import files


#Importo librerías para uso de geopandas
import geopandas as gpd
import requests
import zipfile
import os
import mapclassify

#Importo librerias necesarias para el proceso de Feature engineering y evaluación de modelos
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import roc_curve, auc
import joblib
from pyngrok import ngrok

#Importo librerias necesarias para ML
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from catboost import CatBoostClassifier
import shap
from sklearn.ensemble import RandomForestClassifier
import pwlf

import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)

In [3]:
#Para guardar graficos

os.makedirs("graficos_tfm", exist_ok=True)


In [4]:
import pandas as pd

# LLamo los datos desde Github
url_encuesta = "https://raw.githubusercontent.com/jesustristan17/Exclusion_financiera/refs/heads/main/TMODULO_2024.csv"
url_vivienda = "https://raw.githubusercontent.com/jesustristan17/Exclusion_financiera/refs/heads/main/TVIVIENDA_2024.csv"

# cargo los datos
encuesta_2024 = pd.read_csv(url_encuesta)
#print(encuesta_2024.head())

vivienda_2024 = pd.read_csv(url_vivienda)
#print(vivienda_2024.head())


Column mapping para renombrar las variables de la encuesta a una versión que sea entendible.

In [5]:
column_mapping = {
    "LLAVEVIV": "LLAVEVIV",
    "LLAVEHOG": "LLAVEHOG",
    "LLAVEMOD": "LLAVEMOD",
    "EDAD_V": "EDAD_V",
    "NIV": "ESCOLARIDAD",
    "P3_1A": "AFRODESCENDIENTE",
    "P3_2": "ESTADO_CIVIL",
    "P3_3": "LENGUA_INDIGENA",
    "P3_5_1": "DISC_VISUAL",
    "P3_5_2": "DISC_AUDITIVA",
    "P3_5_3": "DISC_MOTRIZ_SUP",
    "P3_5_4": "DISC_MOTRIZ_INF",
    "P3_5_5": "DISC_CONCENTRACION",
    "P3_5_6": "DISC_OTRA",
    "P3_5_7": "DISC_HABLA",
    "P3_5_8": "DISC_MENTAL",
    "P3_6": "INDIGENA",
    "P3_7": "RAZON_INDIGENA",
    "P3_8": "STATUS_LABORAL",
    "P3_10": "TIPO_TRABAJO",
    "P3_11A": "INGRESO",
    "P3_11B": "PER_INGRESO",
    "P3_12": "TIPO_INGRESO",
    "P3_13": "SEG_SOCIAL",
    "P3_14": "CELULAR",
    "P3_15": "CAMBIO_RES",
    "P3_16": "RAZON_CAMBIO",
    "P4_1": "PRESUPUESTO",
    "P4_2_1": "REGISTRO_GASTOS",
    "P4_2_2": "ORGANIZACION",
    "P4_2_3": "RECIBOS",
    "P4_2_4": "APP_GASTOS",
    "P4_2_5": "DOMICILIACION",
    "P4_3": "INGRESO_SUF",
    "P4_4_1": "PRESTAMO_GASTOS",
    "P4_4_2": "USO_AHORRO",
    "P4_4_3": "REDUCCON_GASTOS",
    "P4_4_4": "EMPEÑAS",
    "P4_4_5": "ADELANTO_SALARIO",
    "P4_4_6": "SOLICITO_CREDITO_F",
    "P4_4_7": "ATRASO_PAGO",
    "P4_9_1": "OPORTUNIDAD_AHORROS",
    "P4_9_2": "OPORTUNIDAD_CREDITO_FORMAL",
    "P4_9_3": "OPORTUNIDAD_EMPEÑO",
    "P4_9_4": "OPORTUNIDAD_CREDITO_INFORMAL",
    "P4_10": "TIEMPO_AHORROS",
    "P4_11": "AFECTACION_DESASTRES",
    "P5_1_1": "AHORRO_PRESTANDO",
    "P5_1_2": "AHORRO_COMPRANDO",
    "P5_1_3": "CAJA_AHORRO",
    "P5_1_4": "AHORRO_TERCEROS",
    "P5_1_5": "TANDA",
    "P5_1_6": "AHORRO_GUARDANDO",
    "P5_3": "COMPARACION_AHORRO",
    "P5_4_3": "CUENTA_GOB",
    "P5_6_3": "USO_CUENTA_GOB",
    "P5_5_8": "CUENTA_DIGITAL",
    "P5_6_8": "USO_CUENTA_DIGITAL",
    "P5_4_9": "OTRAS_CUENTAS",
    "P5_6_9": "USO_OTRAS_CUENTAS",
    "P5_13": "USO_EFECTIVO",
    "P5_19": "CUENTA_PREVIA",
    "P5_20": "RAZON_NO_CUENTA",
    "P5_21": "RAZON_NOCUENTA_PREV",
    "P6_1_1": "PRESTAMO_LABORAL",
    "P6_1_2": "PRESTAMO_EMPEÑO",
    "P6_1_3": "PRESTAMO_AMIGOS",
    "P6_1_4": "PRESTAMO_FAMILIARES",
    "P6_1_5": "PRESTAMO_INFORMALES_OTROS",
    "P6_2_7": "CREDITO_GRUPAL",
    "P6_2_8": "CREDITO_DIGITAL",
    "P6_13": "PRESTAMOS_PREVIOS",
    "P6_14": "RAZON_NOPRESTAMOS",
    "P6_15": "RAZON_NOPRESTAMOS_PREV",
    "P6_16": "RECHAZO_PRESTAMO",
    "P7_1_1": "COMPRAS_INF_500",
    "P7_1_2": "COMPRAS_SUP_500",
    "P7_2_1": "CODI",
    "P7_2_2": "DIMO",
    "P7_10": "PAGOS_TARJETA",
    "P8_2": "SEGURO_PREV",
    "P8_3": "RAZON_NOSEG",
    "P8_4": "RAZON_NOSEG_PREV",
    "P9_1A": "AFORE_PREV",
    "P9_2": "RAZON_NO_AFORE",
    "P10_1": "USO_SUCURSALES",
    "P10_2": "RAZON_NOUSO_SUC",
    "P10_3_1": "DISTANCIA_SUC_HRS",
    "P10_3_2": "DISTANCIA_SUC_MIN",
    "P10_4": "USO_ATM",
    "P10_5": "RAZON_NOUSO_ATM",
    "P10_6_1": "DISTANCIA_ATM_HRS",
    "P10_6_2": "DISTANCIA_ATM_MIN",
    "P10_7": "USO_ALIANZAS",
    "P10_8": "RAZON_NOUSO_ALIANZAS",
    "P10_9_1": "DISTANCIA_ALIANZAS_HRS",
    "P10_9_2": "DISTANCIA_ALIANZAS_MIN",
    "P11_1_1": "ACCESOS_INFO",
    "P11_1_2": "SOLUCIONES",
    "P11_1_3": "DINERO_SEGURO",
    "P11_1_4": "QUEJAS",
    "P11_1_5": "DATOS_PERSONALES",
    "P11_2_1": "CLONACION_TARJETAS",
    "P11_2_2": "FRAUDE_ID",
    "P11_2_3": "OFRECIMIENTOS",
    "P13_1": "TOMADOR_DECISIONES",
    "P13_2_1": "PROPIETARIO_VIVIENDA",
    "P13_2_2": "PROPIETARIO_AUTOMOVIL",
    "P13_2_3": "PROPIETARIO_TERREMO",
    "P13_2_4": "PROPIETARIO_OTROS",
    "SEXO": "GENERO",
    "TLOC": "TLOC",
    "REGION": "REGION",
    "P5_4_1": "P5_4_1",
    "P5_4_2": "P5_4_2",
    "P5_4_4": "P5_4_4",
    "P5_4_5": "P5_4_5",
    "P5_4_6": "P5_4_6",
    "P5_4_7": "P5_4_7",
    "P6_2_1": "P6_2_1",
    "P6_2_2": "P6_2_2",
    "P6_2_3": "P6_2_3",
    "P6_2_4": "P6_2_4",
    "P6_2_5": "P6_2_5",
    "P6_2_6": "P6_2_6",
    "P6_2_9": "P6_2_9",
    "P8_1": "P8_1",
    "P9_1": "P9_1"
}


In [6]:
# filtrar solo las columnas incluidas en el column mapping ya que son las que se identificaron como útiles y reproducibles
encuesta_2024 = encuesta_2024[list(column_mapping.keys())]

#Renombrar las columnas
encuesta_2024.rename(columns=column_mapping, inplace=True)
#encuesta_2024.head()

In [7]:
column_mapping2 = {
    "LLAVEVIV": "LLAVEVIV",
    "P0_1": "HABITACIONES",
    "P0_2": "CUARTOS",
    "P0_3": "BAÑOS",
    "P0_4_1": "AUTOMOVIL",
    "P0_4_1A": "NUM_AUTOS",
    "P0_4_2": "INTERNET",
    "P0_4_2A": "MODEM",
    "P1_1": "NUM_PERSONAS",
    "P1_2": "MISMO_GASTO",
    "P1_3": "HOGARES_DIF",
    "ENT": "ENTIDAD"
}


In [8]:
# Filtrar solo las columnas relevantes
vivienda_2024 = vivienda_2024[list(column_mapping2.keys())]

# Renombrar columnas
vivienda_2024.rename(columns=column_mapping2, inplace=True)


In [9]:
# Cruce por LLAVEVIV
df_combinado = pd.merge(encuesta_2024, vivienda_2024, on="LLAVEVIV", how="inner")

print(f"Tamaño después del cruce: {df_combinado.shape}")

# Validar que no haya duplicados
duplicados = df_combinado["LLAVEMOD"].duplicated(keep=False)
num_duplicados = duplicados.sum()

if num_duplicados == 0:
    print("No hay duplicados.")
else:
    print(f"Hay {num_duplicados} registros duplicados.")


Tamaño después del cruce: (13502, 139)
No hay duplicados.


Función para obtener el porcentaje de nulos por columna

In [10]:
def porcentaje_nulos_por_columna(df):
    """
    Calcula el porcentaje de valores nulos por columna en el DataFrame.

    Parámetros:
        df (pd.DataFrame): DataFrame de entrada

    Retorna:
        pd.Series: Porcentaje de nulos por columna, ordenado descendente
    """
    porcentaje = df.isnull().mean() * 100
    return porcentaje[porcentaje > 0].sort_values(ascending=False).round(2)


In [11]:
porcentaje_nulos = porcentaje_nulos_por_columna(df_combinado)
#Se comenta para no saturar la visualización del notebook
# print(porcentaje_nulos)
# print(len(porcentaje_nulos))



Dado que los nulos en la variable ingreso se interpretan como "sin ingreso", esta variable no la voy a eliminar, pero el resto, al tener un alto porcentaje de nulos saldran del analisis

In [12]:
# Identificar columnas con nulos (excepto INGRESO)
columnas_nulas = df_combinado.columns[df_combinado.isnull().any()]
columnas_a_eliminar = [col for col in columnas_nulas if col != "INGRESO"]

# Eliminar columnas con nulos excepto INGRESO
df_sin_nulos = df_combinado.drop(columns=columnas_a_eliminar)

# Resultado (se comenta para no saturar el notebook)
# print(f"Columnas restantes:")
# for col in df_sin_nulos.columns:
#     print(col)

print(f"Número de columnas restantes: {len(df_sin_nulos.columns)}")

Número de columnas restantes: 97


En esta sentencia tomo las variables que participaran de la variable objetivo: Riesgo de exclusión financiera, si en todas toma el valor 2, la variable Riesgo tomara el valor 1, en otro caso será 0

In [13]:
variables_riesgo = [
    "P5_4_1", "P5_4_2", "P5_4_4", "P5_4_5", "P5_4_6", "P5_4_7",
    "P6_2_1", "P6_2_2", "P6_2_3", "P6_2_4", "P6_2_5", "P6_2_6", "P6_2_9",
    "P8_1", "P9_1"
]
df_sin_nulos["RIESGO"] = (df_sin_nulos[variables_riesgo] == 2).all(axis=1).astype(int)
#df_sin_nulos.head()


In [14]:

# Eliminar las variables del DataFrame
df_sin_nulos = df_sin_nulos.drop(columns=variables_riesgo)

# Verificar las columnas restantes, se comenta para no saturar el notebook
# print("Variables restantes:")
# for col in df_sin_nulos.columns:
#     print(col)


**Análisis exploratorio EDA**

En esta sección analizaré las distribuciones para



*   Detectar tendencias
*   Decidir la mejor forma de imputar nulos
*   Validar si existe oportunidad de binarizar variables
*   Confirmar si no existen nulos ocultos





In [15]:
def graficar_distribuciones_individuales(df):
    """
    Genera una gráfica por cada variable del DataFrame,
    usando histogramas para numéricas y barras para categóricas.

    Parámetros:
        df (pd.DataFrame): DataFrame limpio sin nulos
    """
    for col in df.columns:
        plt.figure(figsize=(6, 4))

        if pd.api.types.is_numeric_dtype(df[col]):
            sns.histplot(df[col], bins=30, kde=True, color='steelblue')
            plt.title(f"Distribución de: {col}")
            plt.xlabel(col)
            plt.ylabel("Frecuencia")
        else:
            conteo = df[col].value_counts().sort_index()
            sns.barplot(x=conteo.index.astype(str), y=conteo.values, color='orange')
            plt.title(f"Frecuencia por categoría: {col}")
            plt.xlabel(col)
            plt.ylabel("Cantidad")
            plt.xticks(rotation=45)

        plt.tight_layout()
        plt.savefig(f"graficos_tfm/{col}_exclusion_ind.png", format="png", dpi=300) #Guardar gráficos
        #plt.show() # Se comenta la instrucción show y se añade close para no mostrar los gráficos, pero si se requiere verlo se pueden invertir.
        plt.close()


In [16]:
#Llamado a la función

graficar_distribuciones_individuales(df_sin_nulos)

**Pricipales hallazgos**



*   La variable objetivo se encuentra desbalanceada (30% en Riesgo de exclusión)
*   Algunas variables binarias (1,2) tiene  otros valores, presumiblemente nulos codificados.


*   Algunas variables requerirán binning para validar su distribución condicional con la variable objetivo








**Distribución condicional vs Target**

Esta otra sección servirá para ver distribución condicional de las variables independientes con la variable dependiente (objetivo).

Derivado de las conclusiones del análisis de distribuciones individuales, las variables continuas se discretizarán.

In [17]:
def graficar_bins_optimos_con_rangos(df, target="RIESGO", min_unique=10, num_bins=5):
    """
    Para todas las variables:
    - Continuas se discretizan con qcut() (cuantiles).
    - Categóricas se grafican directamente.
    - Se agrega grupo NULO como categoría explícita.
    - Se genera gráfico apilado con proporción de RIESGO.
    """

    columnas = [
        col for col in df.columns
        if col != target and not col.startswith("LLAVE")
    ]

    for col in columnas:

        df_temp = df[[col, target]].copy()
        df_temp["GRUPO"] = "NULO"
        sin_nulos = df_temp.dropna(subset=[col]).copy()

        try:
            if pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() > min_unique:
                # Binning con qcut
                grupos = pd.qcut(sin_nulos[col], q=num_bins, duplicates="drop")
                etiquetas = grupos.astype(str).unique().tolist()

                sin_nulos["GRUPO"] = grupos.astype(str)
                df_plot = pd.concat([sin_nulos, df_temp[df_temp["GRUPO"] == "NULO"]])

            else:
                # Categórica o discreta
                df_temp[col] = df_temp[col].astype(object).fillna("NULO")
                df_temp["GRUPO"] = df_temp[col].astype(str)
                df_plot = df_temp.copy()

            tabla_plot = pd.crosstab(df_plot["GRUPO"], df_plot[target], normalize="index")

            # Reordenar si hay NULO
            if "NULO" in tabla_plot.index:
                orden = ["NULO"] + [i for i in tabla_plot.index if i != "NULO"]
                tabla_plot = tabla_plot.loc[orden]

            # Gráfico apilado
            tabla_plot.plot(
                kind="bar",
                stacked=True,
                figsize=(7, 4),
                color=["#1f77b4", "#ff7f0e"]
            )
            plt.title(f"{col} – Proporción de RIESGO")
            plt.xlabel("Grupo / Rango")
            plt.ylabel("Proporción")
            plt.legend(title="RIESGO", labels=["0 = NO excluido", "1 = EXCLUIDO"])
            plt.xticks(rotation=45)
            plt.tight_layout()
            plt.savefig(f"graficos_tfm/{col}_exclusion_bin.png", format="png", dpi=300) #Guardar gráficos
            #plt.show() # Se comenta la instrucción show y se añade close para no mostrar los gráficos, pero si se requiere verlo se pueden invertir.
            plt.close()


        except Exception as e:
            print(f"Error en '{col}': {e}")



In [18]:
#Llamado a la función

graficar_bins_optimos_con_rangos(df_sin_nulos)


**Principales Hallazgos**



*   Variables como Edad, ingreso, Cuartos y baños requieren del binning ya que este mejora su poder predictivo
*   Existen variables que tienen el mismo comportamiento debido a que estan relacionadas entre si (aquellas asociadas a discapacidad, propietario de bienes, uso de plataformas de pagos digitales, etc) todas ellas se abordarán en el apartado de feature engineering.



Discretización variable Edad

In [19]:
# Reemplazar valores de 98 con NaN ya que en la documentación ese es el codigo para los nulos.
df_sin_nulos["EDAD_V"] = df_sin_nulos["EDAD_V"].replace(98, np.nan)

# Imputar nulos con la media
edad_media = df_sin_nulos["EDAD_V"].mean()
df_sin_nulos["EDAD_V"] = df_sin_nulos["EDAD_V"].fillna(edad_media)

# Definir cortes
bins = [17.999, 29, 38, 48, 61, edad_media + (edad_media - 17.999)]
labels = [
    "Jóvenes (18–29)",
    "Adultos jóvenes (30–38)",
    "Adultos medios (39–48)",
    "Adultos mayores (49–61)",
    "Adultos en retiro (62+)"
]

# Crear variable categórica EDAD_BINNED
df_sin_nulos["EDAD_BINNED"] = pd.cut(df_sin_nulos["EDAD_V"], bins=bins, labels=labels, include_lowest=True)

# Verificar distribución
#print(df_sin_nulos["EDAD_BINNED"].value_counts(dropna=False))


In [20]:
# Tabla cruzada: proporción de riesgo por grupo de edad
tabla_edad = pd.crosstab(df_sin_nulos["EDAD_BINNED"], df_sin_nulos["RIESGO"], normalize="index")

# Reordenar
orden = [
    "Jóvenes (18–29)",
    "Adultos jóvenes (30–38)",
    "Adultos medios (39–48)",
    "Adultos mayores (49–61)",
    "Adultos en retiro (62+)"
]
tabla_edad = tabla_edad.reindex(orden)

# Gráfico apilado
tabla_edad.plot(
    kind="bar",
    stacked=True,
    figsize=(7, 4),
    color=["#1f77b4", "#ff7f0e"]
)

plt.title("EDAD_BINNED – Proporción de RIESGO")
plt.xlabel("Grupo de edad")
plt.ylabel("Proporción")
plt.legend(title="RIESGO", labels=["0 = NO excluido", "1 = EXCLUIDO"])
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"graficos_tfm/EDAD_BINNED_exclusion.png", format="png", dpi=300) #Guardar gráficos
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los gráficos, pero si se requiere verlo se pueden invertir.
plt.close()



Mismo analisis con ingreso

In [21]:
# Reemplazar 99888 por NaN
df_sin_nulos["INGRESO"] = df_sin_nulos["INGRESO"].replace(99888, np.nan)

# Separar ingresos válidos (no nulos)
ingreso_valido = df_sin_nulos["INGRESO"].dropna()

# Definir rangos para binning
bins = [0, 1200, 2000, 3500, 10000, ingreso_valido.max() + 1]
labels = [
    "Ingreso muy bajo (0–1200)",
    "Ingreso bajo (1201–2000)",
    "Ingreso medio (2001–3500)",
    "Ingreso alto (3501–10000)",
    "Ingreso muy alto (10001+)"
]

# Inicializar variable como 'Sin Ingreso'
df_sin_nulos["INGRESO_BINNED"] = "Sin Ingreso"

# Asignar etiquetas binned solo a registros con ingreso válido
df_sin_nulos.loc[~df_sin_nulos["INGRESO"].isna(), "INGRESO_BINNED"] = pd.cut(
    ingreso_valido,
    bins=bins,
    labels=labels,
    include_lowest=True
).astype(str)


In [22]:
# Tabla de proporciones por ingreso
tabla_ingreso = pd.crosstab(df_sin_nulos["INGRESO_BINNED"], df_sin_nulos["RIESGO"], normalize="index")

# Reordenar para que 'Sin Ingreso' aparezca primero
orden = ["Sin Ingreso"] + [label for label in labels if label in tabla_ingreso.index]
tabla_ingreso = tabla_ingreso.reindex(orden)

# Gráfico apilado
tabla_ingreso.plot(
    kind="bar",
    stacked=True,
    figsize=(7, 4),
    color=["#1f77b4", "#ff7f0e"]
)

plt.title("INGRESO_BINNED – Proporción de RIESGO")
plt.xlabel("Grupo de ingreso")
plt.ylabel("Proporción")
plt.legend(title="RIESGO", labels=["0 = NO excluido", "1 = EXCLUIDO"])
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"graficos_tfm/INGRESO_BINNED_exclusion.png", format="png", dpi=300) #Guardar gráficos
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los graficos, pero si se requiere verlo se pueden invertir.
plt.close()


Análisis Cuartos

In [23]:
# Reemplazar 99 por np.nan
df_sin_nulos["CUARTOS"] = df_sin_nulos["CUARTOS"].replace(99, np.nan)

# Inicializar como "No especificado"
df_sin_nulos["CUARTOS_BINNED"] = "No especificado"

# Crear rangos
df_sin_nulos.loc[
    df_sin_nulos["CUARTOS"].between(1, 3, inclusive="both"),
    "CUARTOS_BINNED"
] = "1 a 3 cuartos"

df_sin_nulos.loc[
    df_sin_nulos["CUARTOS"] == 4,
    "CUARTOS_BINNED"
] = "4 cuartos"

df_sin_nulos.loc[
    df_sin_nulos["CUARTOS"] == 5,
    "CUARTOS_BINNED"
] = "5 cuartos"

df_sin_nulos.loc[
    df_sin_nulos["CUARTOS"] >= 6,
    "CUARTOS_BINNED"
] = "6 o más cuartos"

# Verificar distribución
#print(df_sin_nulos["CUARTOS_BINNED"].value_counts(dropna=False))


In [24]:
# Tabla de proporciones por grupo de cuartos
tabla_cuartos = pd.crosstab(df_sin_nulos["CUARTOS_BINNED"], df_sin_nulos["RIESGO"], normalize="index")

# Reordenar para que 'No especificado' aparezca primero
orden = ["No especificado", "1 a 3 cuartos", "4 cuartos", "5 cuartos", "6 o más cuartos"]
tabla_cuartos = tabla_cuartos.reindex([i for i in orden if i in tabla_cuartos.index])

# Gráfico apilado
tabla_cuartos.plot(
    kind="bar",
    stacked=True,
    figsize=(7, 4),
    color=["#1f77b4", "#ff7f0e"]
)

plt.title("CUARTOS_BINNED – Proporción de RIESGO")
plt.xlabel("Grupo de número de cuartos")
plt.ylabel("Proporción")
plt.legend(title="RIESGO", labels=["0 = NO excluido", "1 = EXCLUIDO"])
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"graficos_tfm/CUARTOS_BINNED_exclusion.png", format="png", dpi=300) #Guardar gráficos
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los graficos, pero si se requiere verlo se pueden invertir.
plt.close()


Análisis baños

In [25]:
# Reemplazar 99 por NaN
df_sin_nulos["BAÑOS"] = df_sin_nulos["BAÑOS"].replace({99: np.nan})

# Inicializar como "No especificado"
df_sin_nulos["BAÑOS_BINNED"] = "No especificado"

# Asignar categorías
df_sin_nulos.loc[df_sin_nulos["BAÑOS"] == 0, "BAÑOS_BINNED"] = "Sin baño"
df_sin_nulos.loc[df_sin_nulos["BAÑOS"] == 1, "BAÑOS_BINNED"] = "1 baño"
df_sin_nulos.loc[df_sin_nulos["BAÑOS"].isin([2, 3, 4]),"BAÑOS_BINNED"] = "2 a 4 baños"
df_sin_nulos.loc[df_sin_nulos["BAÑOS"] == 5, "BAÑOS_BINNED"] = "5 baños"
df_sin_nulos.loc[df_sin_nulos["BAÑOS"] >= 6, "BAÑOS_BINNED"] = "6 o más baños"

# Verificar distribución
#print(df_sin_nulos["BAÑOS_BINNED"].value_counts(dropna=False))


In [26]:
# Tabla cruzada por proporción
tabla_banos = pd.crosstab(df_sin_nulos["BAÑOS_BINNED"], df_sin_nulos["RIESGO"], normalize="index")

# Reordenar para que 'No especificado' aparezca primero
orden = [
    "No especificado",
    "Sin baño",
    "1 baño",
    "2 a 4 baños",
    "5 baños",
    "6 o más baños"
]
tabla_banos = tabla_banos.reindex([i for i in orden if i in tabla_banos.index])

# Gráfico apilado
tabla_banos.plot(
    kind="bar",
    stacked=True,
    figsize=(7, 4),
    color=["#1f77b4", "#ff7f0e"]
)

plt.title("BANOS_BINNED – Proporción de RIESGO")
plt.xlabel("Número de baños")
plt.ylabel("Proporción")
plt.legend(title="RIESGO", labels=["0 = NO excluido", "1 = EXCLUIDO"])
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(f"graficos_tfm/BANOS_BINNED_exclusion.png", format="png", dpi=300) #Guardar gráficos
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los graficos, pero si se requiere verlo se pueden invertir.
plt.close()



**Analisis demografico**

En esta sección se analizará la correlación espacial, con al finalidad de detectar estados donde la incidencia de exclusión financiera sea mayor, y que pueda servir a la interpretación del modelo.

In [27]:
# URL al ZIP del shapefile en GitHub
url = "https://raw.githubusercontent.com/jesustristan17/Exclusion_financiera/main/dest_2010gw_c.zip"
zip_path = "shapefile.zip"
extract_dir = "shapefile_files"

# Descargar el ZIP
response = requests.get(url)
with open(zip_path, "wb") as f:
    f.write(response.content)

# Extraer archivos
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

# Leer el .shp desde la carpeta extraída
shp_file = os.path.join(extract_dir, "dest_2010cw.shp")
gdf = gpd.read_file(shp_file)

# Mostrar información
#gdf.info()


In [28]:
# Diccionario de las claves de Estado
codigo_a_nombre = {
    1: "AGUASCALIENTES", 2: "BAJA CALIFORNIA", 3: "BAJA CALIFORNIA SUR", 4: "CAMPECHE",
    5: "COAHUILA DE ZARAGOZA", 6: "COLIMA", 7: "CHIAPAS", 8: "CHIHUAHUA", 9: "DISTRITO FEDERAL",
    10: "DURANGO", 11: "GUANAJUATO", 12: "GUERRERO", 13: "HIDALGO", 14: "JALISCO",
    15: "MEXICO", 16: "MICHOACAN DE OCAMPO", 17: "MORELOS", 18: "NAYARIT",
    19: "NUEVO LEON", 20: "OAXACA", 21: "PUEBLA", 22: "QUERETARO DE ARTEAGA", 23: "QUINTANA ROO",
    24: "SAN LUIS POTOSI", 25: "SINALOA", 26: "SONORA", 27: "TABASCO", 28: "TAMAULIPAS",
    29: "TLAXCALA", 30: "VERACRUZ DE IGNACIO DE LA LLAVE", 31: "YUCATAN", 32: "ZACATECAS"
}
# Mapear nombres por código
df_datos=df_sin_nulos.copy()
df_datos['ENTIDAD'] = df_datos['ENTIDAD'].map(codigo_a_nombre)

# Convertir a mayúsculas para asegurar integridad del merge
df_datos['ENTIDAD'] = df_datos['ENTIDAD'].str.strip().str.upper()
gdf['ENTIDAD'] = gdf['ENTIDAD'].astype(str).str.strip().str.upper()


In [29]:
# Agrupar por entidad y calcular la proporción de registros con riesgo de exclusión
riesgo_por_entidad = df_datos.groupby('ENTIDAD').agg(
    total=('RIESGO', 'count'),
    con_riesgo=('RIESGO', 'sum')
)
riesgo_por_entidad['RIESGO_P'] = (riesgo_por_entidad['con_riesgo'] / riesgo_por_entidad['total']).round(4)

# Preparar para el merge
riesgo_por_entidad = riesgo_por_entidad.reset_index()[['ENTIDAD', 'RIESGO_P']]

#Merge con cartografía

gdf_cruce = gdf.merge(
    riesgo_por_entidad,
    left_on='ENTIDAD',
    right_on='ENTIDAD',
    how='left'
)

In [30]:
# definir colormap
cmap_continuo = cm.get_cmap('YlOrRd')

# Normalización
norm = Normalize(vmin=gdf_cruce['RIESGO_P'].min(), vmax=gdf_cruce['RIESGO_P'].max())
gdf_cruce['color'] = gdf_cruce['RIESGO_P'].apply(lambda x: cmap_continuo(norm(x)))

# Graficar
fig, ax = plt.subplots(1, 1, figsize=(10, 10))
gdf_cruce.plot(
    color=gdf_cruce['color'],
    edgecolor='black',
    ax=ax
)
sm = cm.ScalarMappable(cmap=cmap_continuo, norm=norm)
sm._A = []
cbar = fig.colorbar(sm, ax=ax, fraction=0.03, pad=0.04)
cbar.set_label('Riesgo de Exclusión Financiera', fontsize=10)
ax.set_axis_off()
plt.tight_layout()
plt.savefig(f"graficos_tfm/mapa_exclusion.png", format="png", dpi=300)
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los graficos, pero si se requiere verlo se pueden invertir.
plt.close()



Existen comportamiento diferentes por regiones del país, que se explican a mayor profundidad en el informe.

**Imputaciones y recodificaciones**

Derivado de los nulos códificados.

In [31]:
# Crear copia de la base
df_sin_nulos_2 = df_sin_nulos.copy()

# ESCOLARIDAD: 99 y NaN,  imputar proporcionalmente
df_sin_nulos_2["ESCOLARIDAD"] = df_sin_nulos_2["ESCOLARIDAD"].replace(99, np.nan)
escolaridad_dist = df_sin_nulos_2["ESCOLARIDAD"].value_counts(normalize=True)
missing_esc = df_sin_nulos_2["ESCOLARIDAD"].isna()
reemplazos_esc = np.random.choice(escolaridad_dist.index, size=missing_esc.sum(), p=escolaridad_dist.values)
df_sin_nulos_2.loc[missing_esc, "ESCOLARIDAD"] = reemplazos_esc

# AFRODESCENCIENTE: 99 y  NaN, imputar por moda
df_sin_nulos_2["AFRODESCENDIENTE"] = df_sin_nulos_2["AFRODESCENDIENTE"].replace(9, np.nan)
moda_afro = df_sin_nulos_2["AFRODESCENDIENTE"].mode(dropna=True)[0]
df_sin_nulos_2["AFRODESCENDIENTE"] = df_sin_nulos_2["AFRODESCENDIENTE"].fillna(moda_afro)

# INDIGENA: 99 y NaN, imputar por moda
df_sin_nulos_2["INDIGENA"] = df_sin_nulos_2["INDIGENA"].replace(9, np.nan)
moda_indigena = df_sin_nulos_2["INDIGENA"].mode(dropna=True)[0]
df_sin_nulos_2["INDIGENA"] = df_sin_nulos_2["INDIGENA"].fillna(moda_indigena)

# TIEMPO_AHORROS: 8 y 9, convertir a 0
df_sin_nulos_2["TIEMPO_AHORROS"] = df_sin_nulos_2["TIEMPO_AHORROS"].replace([8, 9], 0)

# PAGOS_TARJETA: 9, convertir a 0
df_sin_nulos_2["PAGOS_TARJETA"] = df_sin_nulos_2["PAGOS_TARJETA"].replace(9, 0)

# Otras variables: 9, convertir a 0
variables_nueve = [
    "ACCESOS_INFO", "SOLUCIONES", "DINERO_SEGURO", "QUEJAS",
    "DATOS_PERSONALES", "CLONACION_TARJETAS", "FRAUDE_ID", "OFRECIMIENTOS"
]

for var in variables_nueve:
    df_sin_nulos_2[var] = df_sin_nulos_2[var].replace(9, 2)


In [32]:
#Grafico para ver de nuevo las distribuciones condicionales con los cambios y mejoras hechos
graficar_bins_optimos_con_rangos(df_sin_nulos_2)


In [33]:
# Crear nuevo DataFrame excluyendo las columnas no binarizadas
columnas_a_remover = ["BAÑOS", "CUARTOS", "INGRESO", "EDAD_V"]
df_sin_nulos_3 = df_sin_nulos_2.drop(columns=columnas_a_remover)


**Feature engineering**

En esta sección se haran cambios y agrupamientos de variables que de acuerdo al EDA apoyaran a reducir dimensionalidad y mejorar los resultados del modelo

In [34]:
# Feature sintético - DISCAPACIDAD: Agrupa todas las variables de discapacidad en una sola
df_sin_nulos_3["DISCAPACIDAD"] = (
    df_sin_nulos_3.filter(like="DISC")
    .apply(lambda row: row.isin([3, 4]).any(), axis=1)
    .astype(int)
)

# Feature sintético: OPORTUNIDAD_COMPRA: Agrupa todas las variables relacionadas a como el encuestado afrontaria una oportunidad de adquirir un bien
df_sin_nulos_3["OPORTUNIDAD_COMPRA"] = (
    df_sin_nulos_3[[
        "OPORTUNIDAD_AHORROS",
        "OPORTUNIDAD_CREDITO_FORMAL",
        "OPORTUNIDAD_EMPEÑO",
        "OPORTUNIDAD_CREDITO_INFORMAL"
    ]]
    .eq(1).any(axis=1)
    .astype(int)
)

# Feature sintético - AHORRO_INFORMAL: Agrupa todas las variables relacionadas a ahorro informal
df_sin_nulos_3["AHORRO_INFORMAL"] = (
    df_sin_nulos_3[[
        "AHORRO_PRESTANDO",
        "AHORRO_COMPRANDO",
        "CAJA_AHORRO",
        "AHORRO_TERCEROS",
        "TANDA",
        "AHORRO_GUARDANDO"
    ]]
    .eq(1).any(axis=1)
    .astype(int)
)

# Feature sintético - PRESTAMO_INFORMAL: Agrupa todas las variables relacionada a prestamo informal
df_sin_nulos_3["PRESTAMO_INFORMAL"] = (
    df_sin_nulos_3[[
        "PRESTAMO_LABORAL",
        "PRESTAMO_EMPEÑO",
        "PRESTAMO_AMIGOS",
        "PRESTAMO_FAMILIARES",
        "PRESTAMO_INFORMALES_OTROS"
    ]]
    .eq(1).any(axis=1)
    .astype(int)
)

# Feature sintético - PAGOS_DIGITALES: Agrupa todas las variables relacionada a conocimiento y uso de pagos digitales
df_sin_nulos_3["PAGOS_DIGITALES"] = (
    df_sin_nulos_3[["CODI", "DIMO"]]
    .eq(1).any(axis=1).astype(int)
)

# Feature sintético - PROPIETARIO: Agrupa todas las variables que indican que el encuestado es propietario de un bien
df_sin_nulos_3["PROPIETARIO"] = (
    df_sin_nulos_3[[
        "PROPIETARIO_VIVIENDA",
        "PROPIETARIO_AUTOMOVIL",
        "PROPIETARIO_TERREMO",
        "PROPIETARIO_OTROS"
    ]]
    .eq(1).any(axis=1)
    .astype(int)
)


In [35]:
# Eliminar todas las variables utilizadas para la creación de los features sinteticos
vars_usadas = []

# Para DISCAPACIDAD
vars_usadas += [
    "DISC_VISUAL", "DISC_AUDITIVA",
    "DISC_MOTRIZ_SUP", "DISC_MOTRIZ_INF",
    "DISC_CONCENTRACION", "DISC_OTRA",
    "DISC_HABLA", "DISC_MENTAL"
]

# Para OPORTUNIDAD_COMPRA
vars_usadas += [
    "OPORTUNIDAD_AHORROS", "OPORTUNIDAD_CREDITO_FORMAL",
    "OPORTUNIDAD_EMPEÑO", "OPORTUNIDAD_CREDITO_INFORMAL"
]

# Para AHORRO_INFORMAL
vars_usadas += [
    "AHORRO_PRESTANDO", "AHORRO_COMPRANDO", "CAJA_AHORRO",
    "AHORRO_TERCEROS", "TANDA", "AHORRO_GUARDANDO"
]

# Para PRESTAMO_INFORMAL
vars_usadas += [
    "PRESTAMO_LABORAL", "PRESTAMO_EMPEÑO", "PRESTAMO_AMIGOS",
    "PRESTAMO_FAMILIARES", "PRESTAMO_INFORMALES_OTROS"
]

# Para PAGOS_DIGITALES
vars_usadas += ["CODI", "DIMO"]

# Para PROPIETARIO
vars_usadas += [
    "PROPIETARIO_VIVIENDA", "PROPIETARIO_AUTOMOVIL",
    "PROPIETARIO_TERREMO", "PROPIETARIO_OTROS"
]

# Eliminar duplicados por si alguna se repite
vars_usadas = list(set(vars_usadas))

# Crear nuevo dataframe sin esas variables
df_final_modelo = df_sin_nulos_3.drop(columns=vars_usadas)

# Verificar forma, esta es la dimencionalidad del dataset final
print(f"df_final_modelo tiene {df_final_modelo.shape[0]} filas y {df_final_modelo.shape[1]} columnas.")


df_final_modelo tiene 13502 filas y 60 columnas.


**Selección de variables**

Se utilizará el criterio de IV para determinar las variables que ingresarán al modelo, pero también se considerará limitar el numero de features para que la puesta en producción sea más eficiente

In [36]:
# Función para el IV
def calcular_iv(df, target="RIESGO", excluir=[]):
    iv_dict = {}

    for col in df.columns:
        if col == target or col in excluir:
            continue

        # Convertir a string para tratar todo como categórico
        serie = df[col].astype(str)
        cruzada = pd.crosstab(serie, df[target])

        # Evitar columnas sin ambas clases
        if cruzada.shape[1] < 2:
            continue

        total_0 = cruzada[0].sum()
        total_1 = cruzada[1].sum()

        cruzada["dist_0"] = cruzada[0] / total_0
        cruzada["dist_1"] = cruzada[1] / total_1

        cruzada["woe"] = np.log((cruzada["dist_1"] + 1e-6) / (cruzada["dist_0"] + 1e-6))
        cruzada["iv"] = (cruzada["dist_1"] - cruzada["dist_0"]) * cruzada["woe"]

        iv_dict[col] = cruzada["iv"].sum()

    iv_df = pd.DataFrame.from_dict(iv_dict, orient="index", columns=["IV"]).sort_values("IV", ascending=False)
    return iv_df


In [37]:
# Cálcular IV para todas las variables excepto las que empiezan con 'LLAVE'
iv_resultados = calcular_iv(
    df_final_modelo,
    target="RIESGO",
    excluir=[col for col in df_final_modelo.columns if col.startswith("LLAVE")]
)

IV_df = iv_resultados.reset_index()
IV_df.columns = ['Variable', 'IV']
IV_df = IV_df.sort_values('IV', ascending=False)
#print(IV_df) # se comenta para no hacer el display, pero si se requiere se puede descomentar


In [38]:
#Dado que más de 20 features resultaria en complicar de más el despliegue y que ademas el resto de variables se encuentran por debajo del 0.2 en IV

# Filtrar las 20 variables mas importantes (con el IV)
variables_utiles = iv_resultados[iv_resultados["IV"] > 0.21].index.tolist()

# Incluir también la variable objetivo 'RIESGO' en el nuevo DataFrame
df_iv_filtrado = df_final_modelo[variables_utiles + ["RIESGO"]]


In [39]:
# Seleccionar top 20 variables por IV
top20_df = IV_df.nlargest(20, 'IV')

# Graficar
plt.figure(figsize=(10, 6))
plt.barh(top20_df['Variable'], top20_df['IV'], color='darkcyan')
plt.xlabel('Information Value (IV)')
plt.title('Top 20 Variables por Information Value')
plt.grid(axis='x', linestyle='--', alpha=0.5)
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f"graficos_tfm/Information_value.png", format="png", dpi=300)
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los graficos, pero si se requiere verlo se pueden invertir.
plt.close()


Encoding: En esta etapa se procede con la transformación de features de tipo caracter a numericos para ser considerados inputs en los modelos

También se hace el análisis de correlaciones para evitar multicolineadidad entre variables que puedan afectar el desempeño de los modelos eliminando variables redundantes.

In [40]:
# Crear copia del DataFrame para empezar el codificado
df_encoded = df_iv_filtrado.copy()

encoders = {}

# Uso de label encoder y guardar como pkl en caso de que se requeira después
for col in df_encoded.select_dtypes(include=["category", "object"]).columns:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    encoders[col] = le
    joblib.dump(le, f"label_encoder_{col.lower()}.pkl")
    print(f" Guardado: label_encoder_{col.lower()}.pkl")


# Conversión de binarias 1/2 a 1/0
for col in df_encoded.columns:
    vals = df_encoded[col].dropna().unique()
    if set(vals) == set([1, 2]):
        df_encoded[col] = df_encoded[col].replace({2: 0})

# Excluir variables que comienzan con "LLAVE"
columnas_utiles = [col for col in df_encoded.columns if not col.startswith("LLAVE")]

# Calcular matriz de correlaciones
corr_matrix = df_encoded[columnas_utiles].corr()

# Graficar matriz de correlaciones
plt.figure(figsize=(14, 12))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    cbar_kws={"label": "Correlación"}
)

plt.title("Matriz de Correlación", fontsize=16)
plt.xticks(rotation=45, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("matriz_correlacion.png", dpi=300)
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los graficos, pero si se requiere verlo se pueden invertir.
plt.close()



 Guardado: label_encoder_ingreso_binned.pkl
 Guardado: label_encoder_baños_binned.pkl


No se encontraron correlaciones relevantes, estas 20 variables serán consideradas en la etapa de modelado.

**Division train/test**

In [41]:
# Separar features y target
X = df_encoded.drop(columns=["RIESGO"])
y = df_encoded["RIESGO"]

# División en train/test (estratificado - evitar sobre ajuste por desbalance de clases)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

**Modelado de los datos**

En esta sección se ejecutarán varios modelos de clasificación estudiados durante le master, seleccionando el mejor para la implementación

In [42]:
# Lista de modelos
models = [
    LogisticRegression(max_iter=1000, random_state=42),
    KNeighborsClassifier(),
    DecisionTreeClassifier(random_state=42),
    RandomForestClassifier(random_state=42),
    BaggingClassifier(random_state=42),
    AdaBoostClassifier(random_state=42),
    GradientBoostingClassifier(random_state=42),
    CatBoostClassifier(random_state=42, verbose=0),
    HistGradientBoostingClassifier(random_state=42),
    XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
]

# Nombres de los modelos
names = [
    'Logistic Regression',
    'KNN Classifier',
    'Decision Tree Classifier',
    'Random Forest Classifier',
    'Bagging Classifier',
    'AdaBoost Classifier',
    'Gradient Boosting Classifier',
    'Catboost Classifier',
    'Hist Gradient Boosting',
    'XGBoost Classifier'
]

# Inicializar listas para resultados
accuracy = []
std = []

# Validación cruzada
cv = KFold(n_splits=10, shuffle=True, random_state=1)
for model in models:
    n_scores = cross_val_score(model, X, y, scoring='accuracy', cv=cv, n_jobs=1, error_score='raise')
    accuracy.append(np.mean(n_scores))
    std.append(np.std(n_scores))

# Resultados
score_df = pd.DataFrame({
    'Model': names,
    'Accuracy': accuracy,
    'Std': std
})

# Mostrar resultados en orden descendente
score_df_sorted = score_df.sort_values(by='Accuracy', ascending=False)
print(score_df_sorted)



                          Model  Accuracy       Std
7           Catboost Classifier  0.829951  0.006918
6  Gradient Boosting Classifier  0.829655  0.007726
8        Hist Gradient Boosting  0.829581  0.007420
9            XGBoost Classifier  0.820397  0.010803
3      Random Forest Classifier  0.818027  0.006122
5           AdaBoost Classifier  0.817878  0.006937
0           Logistic Regression  0.812398  0.008286
4            Bagging Classifier  0.802918  0.007904
1                KNN Classifier  0.781885  0.007395
2      Decision Tree Classifier  0.754037  0.011189


El mejor modelo fue el gradient boosting classifier, el cual será aplicado y se obtendran sus metricas de validación y la importancia de features.

In [43]:
# Configurar modelo Gradient Boosting
modelo_gb = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)

# Validación cruzada (10 folds estratificados)
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(modelo_gb, X, y, cv=cv, scoring="accuracy")

print("Resultados de validación cruzada:")
print(f"Accuracy por fold: {np.round(scores, 3)}")
print(f"Promedio: {scores.mean():.3f} | STD: {scores.std():.3f}")

# Entrenamiento y ajuste
modelo_gb.fit(X_train, y_train)
y_pred = modelo_gb.predict(X_test)
y_proba = modelo_gb.predict_proba(X_test)[:, 1]

# Métricas
print("\n Accuracy en test:", accuracy_score(y_test, y_pred))
print("\n Classification Report:")
print(classification_report(y_test, y_pred, digits=3))
print("\n Matriz de Confusión:")
print(confusion_matrix(y_test, y_pred))
#print("\nCurva ROC:")


# Curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC Curve (AUC = {roc_auc:.3f})", color="darkorange")
plt.plot([0, 1], [0, 1], linestyle="--", color="navy", label="Azar")
plt.xlabel("Tasa de falsos positivos (FPR)")
plt.ylabel("Tasa de verdaderos positivos (TPR)")
plt.title("Curva ROC – Gradient Boosting")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(f"graficos_tfm/ROC_GB.png", format="png", dpi=300)
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los graficos, pero si se requiere verlo se pueden invertir.
plt.close()

# Feature importance
importances = modelo_gb.feature_importances_
feature_names = X.columns
feat_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# print("\nImportancia de Features:")
# print(feat_df)

# Visualización de feature importance
plt.figure(figsize=(8, 5))
plt.barh(feat_df['Feature'], feat_df['Importance'], color='teal')
plt.xlabel('Importancia')
plt.title('Importancia de características – Gradient Boosting')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(f"graficos_tfm/Feature_importance_GB.png", format="png", dpi=300)
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los graficos, pero si se requiere verlo se pueden invertir.
plt.close()


Resultados de validación cruzada:
Accuracy por fold: [0.826 0.833 0.855 0.832 0.837 0.821 0.813 0.833 0.831 0.825]
Promedio: 0.831 | STD: 0.011

 Accuracy en test: 0.8308034061458719

 Classification Report:
              precision    recall  f1-score   support

           0      0.871     0.895     0.883      1925
           1      0.721     0.671     0.695       776

    accuracy                          0.831      2701
   macro avg      0.796     0.783     0.789      2701
weighted avg      0.828     0.831     0.829      2701


 Matriz de Confusión:
[[1723  202]
 [ 255  521]]


Aunque el modelo tiene una buen rendimiento (.89 auc, 0.83 en test accuracy) la precisión de la clase minoritaria (riesgo de exclusión) y el F1-score indican que el desbalance en clases afecta la efectividad del modelo, por tanto, para reducir esos errores se aplica un modelo essemble con votación suave ponderada con los 4 mejores modelos de tipo boosting.

In [44]:
# Instanciar modelos individuales
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
hgb = HistGradientBoostingClassifier(random_state=42)
cb = CatBoostClassifier(random_state=42, verbose=0)
xgb = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=6,
                    use_label_encoder=False, eval_metric="logloss", random_state=42)

# Ensemble con VotingClassifier (Weighted Soft Voting)
ensemble_model = VotingClassifier(
    estimators=[
        ('GradientBoosting', gb),
        ('HistGradientBoosting', hgb),
        ('CatBoost', cb),
        ('XGBoost', xgb)
    ],
    voting='soft',
    weights=[0.95, 0.85, 0.75, 0.65]
)

# Validación cruzada
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
ensemble_scores = cross_val_score(ensemble_model, X, y, cv=cv, scoring="accuracy")

print("Resultados de Validación Cruzada del Ensemble:")
print(f"Accuracy por fold: {np.round(ensemble_scores, 3)}")
print(f" Promedio: {ensemble_scores.mean():.3f} | STD: {ensemble_scores.std():.3f}")

# Entrenamiento y evaluación
ensemble_model.fit(X_train, y_train)
y_pred = ensemble_model.predict(X_test)

print("\nAccuracy en test:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, digits=3))
print("\nMatriz de Confusión:")
print(confusion_matrix(y_test, y_pred))


Resultados de Validación Cruzada del Ensemble:
Accuracy por fold: [0.833 0.836 0.85  0.83  0.837 0.822 0.821 0.835 0.827 0.826]
 Promedio: 0.832 | STD: 0.008

Accuracy en test: 0.8293224731580896

Classification Report:
              precision    recall  f1-score   support

           0      0.873     0.890     0.881      1925
           1      0.714     0.678     0.695       776

    accuracy                          0.829      2701
   macro avg      0.793     0.784     0.788      2701
weighted avg      0.827     0.829     0.828      2701


Matriz de Confusión:
[[1714  211]
 [ 250  526]]


El modelo essemble no tiene una mejora visible en accuracy, precision o f1_score, sin embargo reduce el error de clasificación en la clase minoritaria (exclusión) y mejora la ROC, por ello se implementará el modelo de essemble con soft-voting

In [45]:
# Obtener probabilidades del ensemble
y_proba = ensemble_model.predict_proba(X_test)[:, 1]

# Curva Roc
fpr, tpr, _ = roc_curve(y_test, y_proba)
roc_auc = auc(fpr, tpr)

# Visualización
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.3f})', color="darkorange")
plt.plot([0, 1], [0, 1], linestyle="--", color="navy", label="Azar")
plt.xlabel("Tasa de Falsos Positivos")
plt.ylabel("Tasa de Verdaderos Positivos")
plt.title("Curva ROC – Ensemble Model")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(f"graficos_tfm/ROC_Ensemble.png", format="png", dpi=300)
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los graficos, pero si se requiere verlo se pueden invertir.
plt.close()


**Despliegue**

In [46]:
# Guardar modelo entrenado para poderlo usar nuevamente sin necesidad de volverlo a entrenar
joblib.dump(ensemble_model, "modelo_exclusion_financiera.pkl")


['modelo_exclusion_financiera.pkl']

Para segmentar por probabilidad el tipo de riesgo y crear un set de recomendaciones por cada uno de ellos, primero se debe de encontrar el punto de corte de la probabilidad para determinar si el encuestado esta en riesgo de exclusion o no.

In [47]:
y_true = y_test
y_scores = ensemble_model.predict_proba(X_test)[:, 1]

# Calcular curva ROC
fpr, tpr, thresholds = roc_curve(y_true, y_scores)

# Índice de Youden
youden_index = tpr - fpr
optimal_threshold = thresholds[np.argmax(youden_index)]

print(f"Punto de corte óptimo (Youden): {optimal_threshold:.3f}")


Punto de corte óptimo (Youden): 0.280


Con etse resultado del índice de Youden se podría deducir que todo aquel con un valor superior al 0.281 estarái excluido, sin embargo no estarian en el mismo nivel alguien con un 0.3 y otro con un 0.9. Definitivamente hay que definir grados o umbrales de excluisión.

Los cuales podrian ser No excluido, Leve riesgo de exclusión, Excluido y Alta exclusión, de acuerdo con ciertos cortes o umbrales de probabilidad.

Se realiza una regresión segmentada para encontrar los cortes optimos para dividir los resultados en 4 grupos distintos

In [48]:
# Predicción de probabilidades
y_proba = ensemble_model.predict_proba(X_test)[:, 1]

resultado = pd.DataFrame({
    'Prob_Clase_1': y_proba
})

y = np.sort(resultado['Prob_Clase_1'].values)
x = np.linspace(0, 1, len(y))

# Ajuste de regresión segmentada con 4 segmentos
model = pwlf.PiecewiseLinFit(x, y)
breaks = model.fit(4)

# Predicción
x_hat = np.linspace(0, 1, 100)
y_hat = model.predict(x_hat)

# Visualización
plt.hist(resultado['Prob_Clase_1'], bins=50, color='skyblue', edgecolor='black')
plt.axvline(x=breaks[1], color='red', linestyle='--', label='Quiebre 1')
plt.axvline(x=breaks[2], color='green', linestyle='--', label='Quiebre 2')
plt.axvline(x=breaks[3], color='purple', linestyle='--', label='Quiebre 3')
plt.title('Distribución de probabilidad con quiebres')
plt.xlabel('Score de exclusión')
plt.ylabel('Frecuencia')
plt.legend()
plt.savefig(f"graficos_tfm/Regresión segmentada.png", format="png", dpi=300)
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los graficos, pero si se requiere verlo se pueden invertir.
plt.close()


De este grafico, aunque los cortes no son exactos, se puede concluir que los cortes inferidos son 0.25, 0.50 y 0.75

Falta determinar las carácteristicas principales de cada grupo y las recomendaciones para cada gardo de exclusión.

Matriz de indicadores por segmento para analisis y definición de recomendaciones (esto se explicará a profundidad en el informe)

In [49]:
# Se usa como base el dataset de test
X_test_df = X_test.copy()

y_proba = ensemble_model.predict_proba(X_test)[:, 1]

# Añadir al dataser las probabilidades de exclusión
X_test_df['Prob_Clase_1'] = y_proba

# Clasificación en rangos de exclusión
def clasificar_grupo(score):
    if score < 0.25:
        return '1. No excluido'
    elif score < 0.50:
        return '2. Riesgo leve de exclusión'
    elif score < 0.75:
        return '3. Excluido financieramente'
    else:
        return '4. Alta exclusión financiera'

X_test_df['grupo_exclusion'] = X_test_df['Prob_Clase_1'].apply(clasificar_grupo)

# Estadísticas por rango
variables_interes = [
    'USO_ATM', 'RECHAZO_PRESTAMO', 'STATUS_LABORAL', 'COMPRAS_SUP_500',
    'INGRESO_BINNED', 'USO_SUCURSALES', 'ESCOLARIDAD', 'PAGOS_TARJETA',
    'CELULAR', 'INTERNET', 'PAGOS_DIGITALES'
]

variables_validas = [v for v in variables_interes if v in X_test_df.columns]

# Agrupación y resumen
resumen = X_test_df.groupby('grupo_exclusion')[variables_validas].mean().round(2)

# Normalización min-max por columna
resumen_normalizado = resumen.copy()
for col in resumen.columns:
    min_val = resumen[col].min()
    max_val = resumen[col].max()
    if max_val != min_val:
        resumen_normalizado[col] = (resumen[col] - min_val) / (max_val - min_val)
    else:
        resumen_normalizado[col] = 0  # Evita división por cero

# Visualización con heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(resumen_normalizado, annot=resumen, fmt=".2f", cmap="YlGnBu", linewidths=0.5)
plt.title("Heatmap por Indicador según Grupo de Exclusión")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(f"graficos_tfm/Heatmap_grupos_exclusion.png", format="png", dpi=300)
#plt.show() # Se comenta la instrucción show y se añade close para no mostrar los graficos, pero si se requiere verlo se pueden invertir.
plt.close()


**Descargar todos las visualizaciones**

In [50]:
shutil.make_archive("graficos_tfm", "zip", "graficos_tfm")

files.download("graficos_tfm.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**Despliegue**

Se crea la app para poder después llevarla streamlit, en esta parte se hacen algunas validaciones en ngrok para ver que la app funciona antes de subirla a github y hacer el comit en streamlit

Todo lo necesario para la creación del app en streamilit está en mi github https://github.com/jesustristan17/Exclusion_financiera


*   Archivo APP.py: Es la app escrita abajo, la cual hace uso del modelo, depsliega las preguntas y permite al usuario seleccionar valores de una lista, y resultando en una probabilidad de exclusión y recomendaciones personalizadas por rango

*   Modelo_exclusion_financiera.pkl: Archivo pickle que permite el despliegue del modelo, pasarle nuevos datos y obtener el resultado sin necesidad de volver a entrenarlo.



In [51]:
import streamlit as st
import pandas as pd
import joblib

# Cargar modelo entrenado
modelo = joblib.load("modelo_exclusion_financiera.pkl")

# Codificaciones
codificaciones = {
    "INGRESO_BINNED": {
        "Sin ingreso":5,
        "Ingreso menor a 1,200 pesos": 4,
        "Ingreso entre 1,201 y 2,000 pesos": 1,
        "Ingreso entre 2,001 y 3,500 pesos": 2,
        "Ingreso entre 3,501 y 10,000 pesos": 0,
        "Ingreso superior a  10,000 pesos": 3
    },
    "BAÑOS_BINNED": {
        "Sin baño": 4,
        "1 baño": 0,
        "2 a 4 baños": 1,
        "5 baños": 2,
        "6 o más baños": 3
    },
    "ESCOLARIDAD": {
        "Ninguno": 0,
        "Preescolar-Kinder": 1,
        "Primaria": 2,
        "Secundaria": 3,
        "Normal básica": 4,
        "Preparatoria": 6,
        "Carrera técnica": 7,
        "Licenciatura": 8,
        "Especialidad": 9,
        "Maestría": 10,
        "Doctorado": 11
    },
    "STATUS_LABORAL": {
        "Empleado": 1,
        "Informal": 2,
        "Desempleado": 3,
        "Negocio propio": 4,
        "Jubilado/pensionado": 5,
        "Estudiante": 6,
        "Ama de casa": 7,
        "No activo por discapacidad/accidente": 8,
        "Otro": 8,
    },
    "TLOC": {
        "100,000 y más habitantes": 1,
        "15,000 a 99,999 habitantes": 2,
        "2,500 a 14,999 habitantes": 3,
        "Menos de 2,500 habitantes": 4,
    },
    "ENTIDAD": {
      "Aguascalientes": 1,
      "Baja California": 2,
      "Baja California Sur": 3,
      "Campeche": 4,
      "Coahuila de Zaragoza": 5,
      "Colima": 6,
      "Chiapas": 7,
      "Chihuahua": 8,
      "Ciudad de México": 9,
      "Durango": 10,
      "Guanajuato": 11,
      "Guerrero": 12,
      "Hidalgo": 13,
      "Jalisco": 14,
      "Estado de México": 15,
      "Michoacán de Ocampo": 16,
      "Morelos": 17,
      "Nayarit": 18,
      "Nuevo León": 19,
      "Oaxaca": 20,
      "Puebla": 21,
      "Querétaro": 22,
      "Quintana Roo": 23,
      "San Luis Potosí": 24,
      "Sinaloa": 25,
      "Sonora": 26,
      "Tabasco": 27,
      "Tamaulipas": 28,
      "Tlaxcala": 29,
      "Veracruz de Ignacio de la Llave": 30,
      "Yucatán": 31,
      "Zacatecas": 32
  }
,
    "TIEMPO_AHORROS": {
        "No sabe":0,
        "Menos de una semana/ No tiene ahorros": 1,
        "Al menos una semana, pero menos de un mes": 2,
        "Al menos un mes, pero menos de tres meses": 3,
        "Al menos tres meses, pero menos de seis meses": 4,
        "Seis meses o más": 5,
    },
    "COMPRAS_INF_500": {
        "Transferencia electrónica o aplicación de celular": 1,
        "Uso físico de tarjeta de débito o crédito": 2,
        "Efectivo": 3,
    },
        "COMPRAS_SUP_500": {
        "Transferencia electrónica o aplicación de celular": 1,
        "Uso físico de tarjeta de débito o crédito": 2,
        "Efectivo": 3,
    },
        "RECHAZO_PRESTAMO": {
        "Si": 1,
        "No": 2,
        "Nunca lo he solicitado": 3,
    },
        "PAGOS_TARJETA": {
        "En todos": 1,
        "En la mayoría": 2,
        "En algunos": 3,
        "En pocos": 4,
        "En ninguno": 5,
        "No sé":0
    }
}

# Descripciones de cada pregunta
descripciones = {
    "USO_ATM": "En el último año, ¿Has utilizado cajeros automáticos para retirar o consultar saldo?",
    "COMPRAS_SUP_500": "Cuándo realizas compras mayores a $500, ¿Con qué metodo las pagas?",
    "INGRESO_BINNED": "Selecciona el rango que describe tu ingreso mensual aproximado.",
    "ESCOLARIDAD": "Indica el nivel máximo de estudios que has alcanzado.",
    "RECHAZO_PRESTAMO": "¿Alguna vez te han rechazado una solicitud de préstamo?",
    "USO_SUCURSALES": "En el último año ¿has utilizado alguna sucursal de un banco o institución financiera?",
    "COMPRAS_INF_500": "Cuándo realizas compras menores a $500, ¿Con qué metodo las pagas?",
    "STATUS_LABORAL": "Selecciona la opción que mejor describe tu situación laboral actual.",
    "PAGOS_TARJETA": "En los lugares que regularmente compras, ¿En cuántos de ellos aceptan pago con tarjeta o transferencia?",
    "DOMICILIACION": "¿Tienes servicios como luz, agua o internet domiciliados a tu cuenta?",
    "OFRECIMIENTOS": "¿Has recibido ofertas de productos financieros (como tarjetas o seguros)?",
    "TLOC": "Aproximadamente, cuantos habitantes tiene el municipio en el que actualmente vives",
    "CELULAR": "¿Tienes acceso a un teléfono inteligente?",
    "INTERNET": "¿Cuentas con acceso a internet en tu hogar o celular?",
    "CLONACION_TARJETAS": "¿Has experimentado clonación de tarjeta alguna vez?",
    "ENTIDAD": "Selecciona el estado o entidad federativa donde vives.",
    "TIEMPO_AHORROS": "Si dejaras de recibir ingresos, ¿por cuánto tiempo podrías cubrir tus gastos con tus ahorros?",
    "BAÑOS_BINNED": "Número de baños disponibles en tu vivienda.",
    "PAGOS_DIGITALES": "¿Utilizas plataformas digitales para realizar pagos? (CODI, DIMO)",
    "USO_ALIANZAS": "En el último años, ¿has hecho pagos de servicios o depositos a cuentas en tiendas de conveniencia como Oxxo, 7-eleven o supermercados?"
}

# Orden de columnas según entrenamiento
orden_columnas = [
    "USO_ATM",
    "COMPRAS_SUP_500",
    "INGRESO_BINNED",
    "ESCOLARIDAD",
    "RECHAZO_PRESTAMO",
    "USO_SUCURSALES",
    "COMPRAS_INF_500",
    "STATUS_LABORAL",
    "PAGOS_TARJETA",
    "DOMICILIACION",
    "OFRECIMIENTOS",
    "TLOC",
    "CELULAR",
    "INTERNET",
    "ENTIDAD",
    "TIEMPO_AHORROS",
    "BAÑOS_BINNED",
    "PAGOS_DIGITALES",
    "USO_ALIANZAS",
    "CLONACION_TARJETAS"
]


# Configuración de la app
st.set_page_config(page_title="Evaluación Financiera", layout="centered")
st.title("🔍 Evaluación de Exclusión Financiera")

# Entrada de nombre
nombre_usuario = st.text_input(" Ingresa tu nombre ", "")
if nombre_usuario:
    st.write(f"Hola, **{nombre_usuario}** 👋 Bienvenido a la evaluación de exclusión financiera.")
else:
    st.write("Bienvenido a la evaluación de exclusión financiera.")

st.write("Completa el formulario para estimar tu probabilidad de exclusión del sistema financiero.")
# Inputs categóricos codificados
df_input = {}
for var in orden_columnas:
    if var in codificaciones:
        st.write(f"**{var.replace('_', ' ').title()}**")
        st.caption(descripciones.get(var, ""))
        seleccion = st.selectbox("", list(codificaciones[var].keys()), key=var)
        df_input[var] = codificaciones[var][seleccion]

# Inputs binarios tipo Sí/No
binarios_si_no = [
    "USO_ATM", "USO_SUCURSALES",
    "DOMICILIACION", "OFRECIMIENTOS", "CELULAR", "INTERNET", "CLONACION_TARJETAS",
    "PAGOS_DIGITALES", "USO_ALIANZAS"
]


for var in binarios_si_no:
    if var not in df_input:
        st.write(f"**{var.replace('_', ' ').title()}**")
        st.caption(descripciones.get(var, ""))
        valor = st.selectbox("", ["Sí", "No"], key=var)
        df_input[var] = 1 if valor == "Sí" else 0

# Construir DataFrame y asegurar orden correcto
df_input = pd.DataFrame([df_input])
df_input = df_input[orden_columnas]

# Predicción con interpretación de riesgo y acciones
if st.button("Calcular probabilidad"):
    proba = modelo.predict_proba(df_input)[0][1]
    st.metric("Probabilidad de exclusión financiera", f"{proba:.2%}")

    # Interpretación por rangos
    if proba <= 0.25:
        estado = "✅ No excluido"
        detalle = "Tu perfil muestra acceso financiero adecuado."
    elif proba <= 0.50:
        estado = "🟡 Riesgo leve de exclusión"
        detalle = "Presentas algunas barreras financieras que podrían limitar tu acceso."
    elif proba <= 0.75:
        estado = "🟠 Excluido financieramente"
        detalle = "Tu acceso a servicios financieros es limitado. Requiere atención."
    else:
        estado = "🔴 Alta exclusión financiera"
        detalle = "Tu perfil refleja una alta probabilidad de estar excluido del sistema financiero."

    # Mostrar interpretación
    st.subheader("Interpretación")
    st.success(estado)
    st.write(detalle)

    # Acciones recomendadas
    st.markdown("### 🧭 ¿Qué puedes hacer para mejorar tu situación financiera?")
    if proba <= 0.25:
        st.markdown("""
            Tu acceso financiero es adecuado. ¡Bien hecho!
            - Sigue usando los servicios que ya tienes (cuentas, cajeros, pagos digitales).
            - Solo repite esta evaluación si cambias de trabajo, tus ingresos bajan o tu situación personal cambia.
        """)
    elif proba <= 0.50:
        st.markdown("""
            Estás en una etapa temprana de riesgo. Es buen momento para actuar:
            - Aprende más sobre cómo manejar tu dinero. Hay cursos gratuitos en línea y en tu comunidad.
            - Si necesitas un préstamo, busca opciones que se puedan solicitar desde el celular, sin ir al banco.
            - Usa cajeros automáticos cuando puedas, y si no hay cerca, pregunta por cajeros móviles o tiendas que den servicios financieros.
            - Repite esta evaluación dentro de **1 año** para ver si has mejorado.
        """)
    elif proba <= 0.75:
        st.markdown("""
            Tienes acceso limitado a servicios financieros. Hay formas de avanzar:
            - Pregunta en tu trabajo o comunidad si hay programas para abrir cuentas bancarias básicas.
            - Aprende a usar apps para pagar, ahorrar o enviar dinero. Muchas son fáciles y seguras.
            - Busca cuentas que premien el uso digital (como no cobrar comisiones si usas la app).
            - Repite esta evaluación dentro de **6 meses** para revisar tu progreso.
        """)
    else:
        st.markdown("""
            Tu situación muestra una alta exclusión financiera. No estás solo, y hay formas de empezar:
            - Acércate a programas sociales (educación, salud, empleo) que también ayudan a abrir cuentas bancarias.
            - Pregunta por cuentas sin comisiones que se puedan abrir en persona, sin necesidad de internet.
            - Si no tienes celular o internet, busca centros comunitarios donde puedas conectarte o recibir ayuda.
            - Participa en talleres o apoyos para aprender sobre dinero, ahorro y pagos digitales.
            - Repite esta evaluación dentro de **3 meses** para seguir tu avance.
        """)


2025-09-08 22:33:08.124 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-08 22:33:08.128 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-08 22:33:08.378 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2025-09-08 22:33:08.379 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-08 22:33:08.380 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-08 22:33:08.382 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-09-08 22:33:08.384 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [52]:
from pyngrok import ngrok
ngrok.set_auth_token("304H6US24qKSn2BPVs8Zwywoyex_5k8az8SAkSf2Ch5ZURVbz")


In [53]:
!tail -n 50 /content/log.txt


tail: cannot open '/content/log.txt' for reading: No such file or directory


In [54]:
!streamlit run app.py &>/content/log.txt &

# Abrir túnel web
public_url = ngrok.connect(addr="8501", proto="http")
print("App está disponible en:", public_url)


App está disponible en: NgrokTunnel: "https://0aaedd5c7b4a.ngrok-free.app" -> "http://localhost:8501"


Ruta de la app en Streamlit: https://exclusionfinanciera-c7ns2wpsxhfefszqjdhwze.streamlit.app/